# Default Orchestrator Demo (CVAE + Chemist Agent V1 + ESM Z-score Biologist)

This notebook demonstrates an end-to-end pipeline:

1. Load training data from `database/training_data.json` with the project dataloader.
2. Train the `CVAEGenerator`.
3. Build an orchestrator using:
   - `ChemistAgent` from `agent_v1`
   - `ESMBiologistZscore`
   - the trained `CVAEGenerator`
4. Run iterative optimization.
5. Attempt to compare generated peptides against baseline outputs.

In [1]:
import os
import sys
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "orchestrator":
    REPO_ROOT = REPO_ROOT.parent.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"Torch device availability: cuda={torch.cuda.is_available()}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    torch.random.manual_seed(SEED)
except Exception:
    pass

Repo root: /home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline
Torch device availability: cuda=True


In [2]:
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader

DATA_PATH = REPO_ROOT / "database" / "training_data.json"

loader = JSONDataLoader()
loader.load_data(
    source=str(DATA_PATH),
    columns=[
        "sequence",
        "length",
        "ph",
        "molecular_weight",
        "logp",
        "net_charge",
        "isoelectric_point",
        "hydrophobicity",
        "cathionicity",
    ],
    fillna_defaults={
        "length": 10,
        "ph": 7.0,
        "molecular_weight": 1500.0,
        "logp": 0.0,
        "net_charge": 0.0,
        "isoelectric_point": 7.0,
        "hydrophobicity": 0.0,
        "cathionicity": 0.0,
    },
    normalize_sequence=True,
    sequence_column="sequence",
    keep_standard_amino_acids_only=True,
    length_column="length",
    length_range=(1, 14),
)

df = loader.get_data().copy()

print(f"Loaded records after dataloader preprocessing: {len(df)}")
print("NaN count per column:")
print(df.isna().sum())
display(df.head(3))

Loaded records after dataloader preprocessing: 3653
NaN count per column:
sequence             0
length               0
ph                   0
molecular_weight     0
logp                 0
net_charge           0
isoelectric_point    0
hydrophobicity       0
cathionicity         0
dtype: int64


,sequence,length,ph,molecular_weight,logp,net_charge,isoelectric_point,hydrophobicity,cathionicity
0,KVVVKWVVKVVK,12,7.0,1648.291,5.6026,5.0,14.0,-1.07,4
1,LFIFFF,6,7.0,832.059,3.2860,1.0,14.0,-3.25,0
2,KAAAKWAAKAAK,12,7.0,1451.913,1.1499,5.0,14.0,0.33,4


In [3]:
from peptide_pipeline.generator.cvae_generator import CVAEGenerator

AA = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {aa: i for i, aa in enumerate(AA)}
PAD_IDX = 20
VOCAB_SIZE = 21
MAX_LEN = 14


def encode_one_hot_with_pad(sequences, max_len=MAX_LEN):
    x = torch.zeros(len(sequences), max_len * VOCAB_SIZE, dtype=torch.float32)
    for i, seq in enumerate(sequences):
        for pos in range(max_len):
            x[i, pos * VOCAB_SIZE + PAD_IDX] = 1.0
        for pos, aa in enumerate(seq[:max_len]):
            if aa in AA_TO_IDX:
                x[i, pos * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, pos * VOCAB_SIZE + AA_TO_IDX[aa]] = 1.0
    return x


def build_condition_tensor(dataframe, model):
    # Datalaoder already normalizes and fills defaults. We only map baseline fields
    # into the condition layout expected by CVAEGenerator.
    required_cols = [
        "length",
        "molecular_weight",
        "net_charge",
        "isoelectric_point",
        "hydrophobicity",
        "cathionicity",
        "logp",
    ]
    missing = [c for c in required_cols if c not in dataframe.columns]
    if missing:
        raise ValueError(f"Missing required columns for conditioning: {missing}")

    c_df = dataframe[required_cols].copy()
    if c_df.isna().any().any():
        raise ValueError("NaN values detected after dataloader preprocessing.")

    cond = torch.zeros(len(c_df), model.condition_dim, dtype=torch.float32)

    # CVAE property order:
    # [size, molecular_weight, net_charge_pH5_5, isoelectric_point, hydrophobicity,
    #  cathionicity, hydrophobic_moment, logp, boman_index, h_bond_donors,
    #  h_bond_acceptors, tpsa]
    cond[:, 0] = torch.tensor(c_df["length"].values, dtype=torch.float32)
    cond[:, 1] = torch.tensor(c_df["molecular_weight"].values, dtype=torch.float32)
    cond[:, 2] = torch.tensor(c_df["net_charge"].values, dtype=torch.float32)
    cond[:, 3] = torch.tensor(c_df["isoelectric_point"].values, dtype=torch.float32)
    cond[:, 4] = torch.tensor(c_df["hydrophobicity"].values, dtype=torch.float32)
    cond[:, 5] = torch.tensor(c_df["cathionicity"].values, dtype=torch.float32)
    cond[:, 6] = 0.5
    cond[:, 7] = torch.tensor(c_df["logp"].values, dtype=torch.float32)
    cond[:, 8] = 0.0
    cond[:, 9] = 5.0
    cond[:, 10] = 5.0
    cond[:, 11] = 100.0

    return cond


sequences = df["sequence"].tolist()
lengths = torch.tensor(df["length"].astype(int).values, dtype=torch.long)

cvae = CVAEGenerator(max_len=MAX_LEN, latent_dim=64, hidden_dim=256, condition_dim=32)

x = encode_one_hot_with_pad(sequences, max_len=MAX_LEN)
conditions = build_condition_tensor(df, cvae)

# Move tensors to model device
x = x.to(cvae.device)
conditions = conditions.to(cvae.device)
lengths = lengths.to(cvae.device)

print(f"x shape: {tuple(x.shape)}")
print(f"conditions shape: {tuple(conditions.shape)}")
print(f"lengths shape: {tuple(lengths.shape)}")
print(f"CVAE device: {cvae.device}")
print("Condition columns loaded from baseline schema: length, ph, molecular_weight, logp, net_charge, isoelectric_point, hydrophobicity, cathionicity")

x shape: (3653, 294)
conditions shape: (3653, 32)
lengths shape: (3653,)
CVAE device: cuda
Condition columns loaded from baseline schema: length, ph, molecular_weight, logp, net_charge, isoelectric_point, hydrophobicity, cathionicity


In [4]:
CVAE_OUT = REPO_ROOT / "peptide_pipeline" / "generator" / "cvae_orchestrator_demo.pth"

if CVAE_OUT.exists():
    try:
        cvae.load_model(str(CVAE_OUT))
    except AttributeError:
        state = torch.load(CVAE_OUT, map_location=cvae.device)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        cvae.load_state_dict(state)
        cvae.to(cvae.device)
    print(f"Loaded existing CVAE from: {CVAE_OUT}")
else:
    # Train CVAE on training_data.json via the dataloader output
    cvae.train_model(
        data=x,
        conditions=conditions,
        lengths=lengths,
        epochs=100,
        batch_size=64,
        lr=1e-3,
        kl_anneal_epochs=60,
    )
    cvae.save_model(str(CVAE_OUT))
    print(f"Saved trained CVAE to: {CVAE_OUT}")

Loaded existing CVAE from: /home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline/peptide_pipeline/generator/cvae_orchestrator_demo.pth


In [5]:
from peptide_pipeline.chemist.agent_v1.config_chemist import ChemistConfig, RangeTarget
from peptide_pipeline.chemist.agent_v1.chemist_agent import ChemistAgent
from peptide_pipeline.biologist.esm_biologist_zscore import ESMBiologistZscore
from peptide_pipeline.orchestrator.orchestrator import Orchestrator

# Baseline-like target profile from baseline/inference.py
BASELINE_TARGET = {
    "length": 10.0,
    "ph": 7.0,
    "molecular_weight": 1200.0,
    "logp": 2.5,
    "net_charge": 3.0,
    "isoelectric_point": 10.0,
    "hydrophobicity": -0.5,
    "cathionicity": 3.0,
}

# Build chemist constraints around baseline targets.
# Note: cathionicity is intentionally not constrained because the installed
# peptides package here does not expose Peptide.cathionicity().
chemist_config = ChemistConfig(
    ph=BASELINE_TARGET["ph"],
    length=RangeTarget(min=8.0, max=12.0, target=BASELINE_TARGET["length"], weight=1.0),
    molecular_weight=RangeTarget(min=900.0, max=1500.0, target=BASELINE_TARGET["molecular_weight"], weight=1.0),
    logp=RangeTarget(min=0.0, max=5.0, target=BASELINE_TARGET["logp"], weight=1.0),
    net_charge=RangeTarget(min=1.0, max=6.0, target=BASELINE_TARGET["net_charge"], weight=1.0),
    isoelectric_point=RangeTarget(min=8.0, max=12.0, target=BASELINE_TARGET["isoelectric_point"], weight=1.0),
    hydrophobicity=RangeTarget(min=-2.0, max=2.0, target=BASELINE_TARGET["hydrophobicity"], weight=1.0),
)

# Use a dataset peptide close to baseline targets as ESM reference
score_cols = ["length", "molecular_weight", "isoelectric_point", "hydrophobicity", "cathionicity"]
ref_df = df[score_cols + ["sequence"]].copy()
for col in score_cols:
    std = ref_df[col].std() if ref_df[col].std() != 0 else 1.0
    ref_df[f"z_{col}"] = (ref_df[col] - BASELINE_TARGET[col]) / std
ref_df["proxy_dist"] = (ref_df[[f"z_{c}" for c in score_cols]] ** 2).sum(axis=1) ** 0.5
reference_peptide = ref_df.sort_values("proxy_dist").iloc[0]["sequence"]

print(f"Reference peptide for ESM z-score agent: {reference_peptide}")

chemist = ChemistAgent(config=chemist_config)
biologist = ESMBiologistZscore(reference_peptide=reference_peptide, batch_size=16)

generator_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cvae = cvae.to(generator_device)
cvae.device = generator_device
print(f"Generator device: {cvae.device}")

orchestrator = Orchestrator(generator=cvae, chemist=chemist, biologist=biologist)

/home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reference peptide for ESM z-score agent: ELLKAVRLIK


Loading weights: 100%|██████████| 209/209 [00:00<00:00, 1368.49it/s, Materializing param=encoder.layer.11.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Generator device: cuda


In [6]:
# Run default orchestrator loop
results = orchestrator.run(
    nb_iterations=6,
    nb_peptides=64,
    top_k=10,
    exploration_rate=0.15,
    initial_peptide=reference_peptide,
)

orchestrator_df = pd.DataFrame(results)
if not orchestrator_df.empty:
    display(orchestrator_df)
else:
    print("No orchestrator results were returned.")

orchestrator_peptides = orchestrator_df["peptide"].tolist() if "peptide" in orchestrator_df.columns else []
print("Top orchestrator peptides:")
for i, pep in enumerate(orchestrator_peptides, start=1):
    print(f"{i:02d}. {pep}")

No final_target provided. Using targets inferred from chemist config.


,peptide,score,combined_score,chemist_score,biologist_score,in_limits,properties,iteration
0,YKYKSFLKAW,0.848799,0.848799,0.861183,0.836416,True,"{'length': 10, 'molecular_weight': 1333.59574,...",4
1,FWELKTKVKF,0.834804,0.834804,0.831131,0.838476,True,"{'length': 10, 'molecular_weight': 1325.61654,...",5
2,FPKWLHLKAW,0.833276,0.833276,0.883809,0.782742,True,"{'length': 10, 'molecular_weight': 1325.621839...",6
3,FPLKRWLVAL,0.815190,0.815190,0.781247,0.849132,True,"{'length': 10, 'molecular_weight': 1242.57294,...",2
4,GLLKAWKLAL,0.798160,0.798160,0.763314,0.833006,True,"{'length': 10, 'molecular_weight': 1112.42374,...",4
5,YYVALKIRFP,0.791252,0.791252,0.795070,0.787433,True,"{'length': 10, 'molecular_weight': 1269.55234,...",5
6,FIGFKIFLWR,0.782834,0.782834,0.807047,0.758621,True,"{'length': 10, 'molecular_weight': 1326.64994,...",3
7,FPKHWFYWVK,0.777746,0.777746,0.886664,0.668829,True,"{'length': 10, 'molecular_weight': 1437.709439...",6
8,KGVHLWLKWL,0.772998,0.772998,0.833471,0.712524,True,"{'length': 10, 'molecular_weight': 1279.59364,...",1
9,WWFKAFIAKW,0.769824,0.769824,0.792785,0.746863,True,"{'length': 10, 'molecular_weight': 1382.673239...",5


Top orchestrator peptides:
01. YKYKSFLKAW
02. FWELKTKVKF
03. FPKWLHLKAW
04. FPLKRWLVAL
05. GLLKAWKLAL
06. YYVALKIRFP
07. FIGFKIFLWR
08. FPKHWFYWVK
09. KGVHLWLKWL
10. WWFKAFIAKW


In [7]:
# Try to generate baseline peptides and compare overlap with orchestrator output
import pickle

BASELINE_DIR = REPO_ROOT / "baseline"
if str(BASELINE_DIR) not in sys.path:
    sys.path.insert(0, str(BASELINE_DIR))

baseline_model_path = BASELINE_DIR / "cvae_peptide_model.pth"
baseline_scaler_path = BASELINE_DIR / "scaler.pkl"

baseline_peptides = []

if baseline_model_path.exists() and baseline_scaler_path.exists():
    import data_handler as bl_data_handler
    import model as bl_model
    import inference as bl_inference

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        raw_records = __import__("json").load(f)

    filtered_records = [r for r in raw_records if len(str(r.get("sequence", ""))) <= 12]
    _, vocab_size, condition_dim = bl_data_handler.get_dataloader(
        records=filtered_records,
        batch_size=32,
        max_len=12,
        is_train=False,
        scaler_path=str(baseline_scaler_path),
    )

    baseline_model = bl_model.PeptideCVAE(
        vocab_size=vocab_size,
        condition_dim=condition_dim,
        max_seq_len=14,
        latent_dim=32,
    )
    state = torch.load(baseline_model_path, map_location="cpu")
    baseline_model.load_state_dict(state)

    with open(baseline_scaler_path, "rb") as f:
        baseline_scaler = pickle.load(f)

    baseline_peptides = bl_inference.generate_peptides(
        model=baseline_model,
        scaler=baseline_scaler,
        num_samples=max(10, len(orchestrator_peptides) or 10),
        properties=[10, 7.0, 1200.0, 2.5, 3.0, 10.0, -0.5, 3],
        temperature=1.0,
        top_k=5,
    )
else:
    print("Baseline model/scaler not found. Using baseline-like CVAE-conditioned generation fallback.")
    fallback_constraints = {
        "size": 10,
        "molecular_weight": 1200.0,
        "net_charge_pH5_5": 3.0,
        "isoelectric_point": 10.0,
        "hydrophobicity": -0.5,
        "cathionicity": 3.0,
    }
    baseline_peptides = cvae.generate_peptides(
        count=max(10, len(orchestrator_peptides) or 10),
        constraints=fallback_constraints,
        temperature=0.9,
    )

print("Baseline peptides (or fallback):")
for i, pep in enumerate(baseline_peptides[:20], start=1):
    print(f"{i:02d}. {pep}")

set_orch = set(orchestrator_peptides)
set_base = set(baseline_peptides)
overlap = set_orch.intersection(set_base)
overlap_ratio = (len(overlap) / max(1, len(set_orch))) * 100.0

print("\nComparison summary")
print(f"Orchestrator unique peptides: {len(set_orch)}")
print(f"Baseline unique peptides: {len(set_base)}")
print(f"Exact overlap count: {len(overlap)}")
print(f"Overlap wrt orchestrator set: {overlap_ratio:.2f}%")
if overlap:
    print("Overlapping peptides:")
    for pep in sorted(overlap):
        print(f"- {pep}")

Baseline model/scaler not found. Using baseline-like CVAE-conditioned generation fallback.
Baseline peptides (or fallback):
01. FLLKVMKKMK
02. GMLAKFLKKI
03. VKIFLKLHKV
04. FVCKAWLMRH
05. GVLKKLLKFL
06. VLLHLLKTFA
07. FQPFLGLKKR
08. WPLVKLLAKK
09. FKKKLGLWKK
10. KSLKKRLKGF

Comparison summary
Orchestrator unique peptides: 10
Baseline unique peptides: 10
Exact overlap count: 0
Overlap wrt orchestrator set: 0.00%
